[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo4_taller/modulo4_taller.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 4: Aplicación. Reto. Taller.

El taller reúne los módulos anteriores en un flujo completo, con datos que llegan como en la vida
real: en distintos sistemas de coordenadas y en formatos distintos.

Objetivo: producir un mapa de idoneidad para un proyecto eólico en el norte de Colombia.

Pasos:

1. Cargar las fuentes (vienen en distintos CRS y formatos).
2. Unificar: llevar todo al mismo sistema de coordenadas.
3. Generar la malla (H3).
4. Mapear las fuentes sobre la malla.
5. Visualizar.
6. Decidir: combinar los criterios con AHP y obtener el mapa final.

Corre completo en Colab. Los archivos están también en `recursos/`.

## 0. Preparación

In [28]:
!pip install -q geopandas "h3>=4.1" folium mapclassify

In [ ]:
# Solo en Colab: Colab abre unicamente el notebook, no la carpeta recursos/.
# Clonamos el repositorio para tener los datos disponibles y entramos a la carpeta del modulo.
import os
if not os.path.exists("recursos"):
    !git clone --depth 1 https://github.com/Sadriica/Curso_ANH.git
    %cd Curso_ANH/modulo4_taller


In [29]:
import pandas as pd
import numpy as np
import geopandas as gpd
import h3
import folium
import xarray as xr
from rasterstats import zonal_stats
from shapely.geometry import Polygon
from shapely.geometry import Point


## 1. Las fuentes (distintos CRS y formatos)

Fuentes de datos:

- **Velocidad del viento:** ráster en  EPSG:4326.
- **Pendiente:** ráster con la estimación de la pendiente en Colombia en EPSG:9377
- **Especies de aves:** archivo shapefile que contiene la riqueza de aves en lista IUCN en EPSG:9377
- **Índice de desempeño institucional:** csv con el desempeño de las instituciones en un municipio 
- **Índice de pobreza multidimensional:** excel con el índice de pobreza multidimensional en un municipio
- **Municipios de Colombia:** Capa shapefile con los municipios de colombia en EPSG:4326  

Para comenzar, cargaremos los archivos CSV correspondientes al Índice de Desempeño Institucional (IDI) y al Índice de Pobreza Multidimensional (IPM). Ambas fuentes contienen el nombre del municipio, el valor del índice y su código oficial DANE. Utilizaremos este código municipal como identificador único para cruzar e integrar ambos conjuntos de datos. 

In [43]:
# cargar municipios de colombia
mnpios_col=gpd.read_file('recursos/MUNICPIOS_COLOMBIA')
# Hacemos el índice sea el código de cada municipio
mnpios_col.set_index('MpCodigo',drop=True,inplace=True)
# Cargar datos de índice de pobreza
df_ipm = pd.read_excel('recursos/IPM-municipal-valor.xlsx')
# Cargar datos de índice de desempeño institucional
df_idi = pd.read_csv(
    'recursos/desempeno_institucional/desempeno_institucional_final.csv',
    dtype={'codigo_dane': str}, # Le estamos diciendo a pandas que cargue el codigo como un string
    index_col=0 # Le decimos a pandas que la primera columna sea el índice
)

- Códigos de municipio como texto (object): Cargaremos los códigos DANE como texto y no como número para conservar los ceros a la izquierda (por ejemplo, 05001). Si los leemos como enteros, el cero inicial se elimina (5001), lo que rompería la unión entre tablas. Adicionalmente, corregiremos los errores de formato identificados en los códigos del IPM.

- Ajuste de coordenadas (IPM): En la tabla del IPM, las coordenadas utilizan coma (,) en lugar de punto (.) como separador decimal, provocando que Python las interprete como cadenas de texto (object). Reemplazaremos las comas por puntos y transformaremos estas columnas a tipo flotante (float).

In [44]:
codigos = df_ipm['Código Municipio']
# Arreglo de los codigos para que queden en formato str
# Esta lista almacena los codigos corregidos
cods_arreglados = list()
# Ciclo for que va por cada código y revisa si está correcto
for codigo in codigos:
    # si el codigo es de tamaño igual a 4 le agrega el 0
    if len(str(codigo)) == 4:
        codigo_fix = '0' + str(codigo)
        cods_arreglados.append(codigo_fix)
    # Caso contrario solo lo ocnvierte a str
    else:
        codigo_fix = str(codigo)
        cods_arreglados.append(codigo_fix)

# Actualizacion de los codigos
df_ipm['Código Municipio'] = cods_arreglados
# Cambio de indice
df_ipm.set_index('Código Municipio',drop=True,inplace=True)
# Georreferenciacion del indice de pobreza multidimensional #
# Además se observa que las coordenadas están con ',' en lugar de '.'
df_ipm['IPM'] = [float(x.replace(',','.')) for x in df_ipm['IPM']] 
# Relacion del IPM con los municipios
mnpios_col['ipm'] = df_ipm['IPM']
# Georreferenciacion idi
mnpios_col['idi'] = df_idi['desempeno_municipal']

Se cargan las capas de especies de aves IUCN, la pendiente del terreno y la velocidad del viento.

In [45]:
# Cargar datos de especies
gdf_especies = gpd.read_file('recursos/AMB-017-A/AMB-017-A.shp')
# Cargar datos de pendiente
tif_pendiente = xr.open_dataarray('recursos/AMB-007-AT/Pendientes.tif')
# Cargar velocidad del viento
tif_viento = xr.open_dataarray('recursos/Velocidad_100m_patched.tif')

se genera la capa que contiene la ubicación de los municipios de interés sobre los cuales deseamos implementar parques eólicos:
- Uribia: Latitud 11.71, Longitud -71.98

- Riohacha: Latitud 11.55, Longitud -72.91 

- Maicao: Latitud 11.38, Longitud -72.24 

- Manaure: Latitud 11.78, Longitud -72.44 

- Valledupar: Latitud 10.46, Longitud -73.25 

- Aguachica: Latitud 8.31, Longitud -73.63 

- Santa Marta: Latitud 11.24, Longitud -74.20 

- Barranquilla: Latitud 10.96, Longitud -74.80 

In [46]:
# Municipios de interés
base = pd.DataFrame({
    "sitio": ["Uribia", "Riohacha", "Maicao", "Manaure",
              "Valledupar", "Aguachica", "Santa Marta", "Barranquilla"],
    "lat": [11.71, 11.55, 11.38, 11.78, 10.46, 8.31, 11.24, 10.96],
    "lon": [-71.98, -72.91, -72.24, -72.44, -73.25, -73.63, -74.20, -74.80],
})

# Creación de Geodataframe
mnpios = gpd.GeoDataFrame(
    base,
    geometry=gpd.points_from_xy(base["lon"], base["lat"]), crs="EPSG:4326",
)


A cada una de las fuentes chequear su sistema de coordenadas.
Describir sus fuentes coordenadas
¿ Se puede trabajar con los datos cómo se encuentran?

In [47]:
crs_especies = gdf_especies.crs
crs_pendiente = tif_pendiente.rio.crs
crs_viento = tif_viento.rio.crs
crs_idi_idm = mnpios_col.crs

crs_municipios = mnpios.crs
# Revision de las fuentes de los datos
print(f'Sistema coordenado especies aves:{crs_especies}')
print(f'Sistema coordenado pendiente:{crs_pendiente}')
print(f'Sistema coordenado viento:{crs_viento}')
print(f'Sistema coordenado especies municipios interes:{crs_municipios}')
print(f'Sistema coordenado especies IDI, IPM:{crs_idi_idm}')


Sistema coordenado especies aves:EPSG:9377
Sistema coordenado pendiente:PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1329"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","20046"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["central_meridian",-73],PARAMETER["scale_factor",0.9992],PARAMETER["false_easting",5000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","9377"]]
Sistema coordenado viento:EPSG:4326
Sistema coordenado especies municipios interes:EPSG:4326
Sistema coordenado especies IDI, IPM:EPSG:9377


## 2. Unificar: mismo sistema de coordenadas

Las fuentes de datos se encuentran en distintos sistemas coordenados, por lo cual se lleva todo a
EPSG:4326, para facilitar el trabajo con el módulo H3

In [48]:
crs_reference = 'EPSG:4326'
# Modificacion de datos shape
gdf_especies = gdf_especies.to_crs(crs_reference)
mnpios_col = mnpios_col.to_crs(crs_reference)
# Modificacion de datos tif
tif_pendiente = tif_pendiente.rio.reproject(crs_reference)

In [49]:
# Corroborar que se hayan generado las modificaciones
print(f'Sistema coordenado especies aves: {gdf_especies.crs}')
print(f'Sistema coordenado especies IDI, IPM: {mnpios_col.crs}')
print(f'Sistema coordenado pendiente: {tif_pendiente.rio.crs}')

Sistema coordenado especies aves: EPSG:4326
Sistema coordenado especies IDI, IPM: EPSG:4326
Sistema coordenado pendiente: EPSG:4326


Una vez validados los datos y alineados bajo el mismo sistema de referencia espacial (SRC / CRS), asociaremos las variables a cada municipio de interés.
Para mantener una estructura ordenada, agruparemos la información en dos conjuntos de datos:
- fuente_a (Variables físico-ambientales): Velocidad del viento, pendiente y presencia de aves.
- fuente_b (Variables socioeconómicas e institucionales): Índice de Desempeño Institucional (IDI) e Índice de Pobreza Multidimensional (IPM).

In [50]:
# Extraccion de datos para cada municipio
fuente_a = base.copy()
for idx in base.index:
    lat = base.iloc[idx,1]
    lon = base.iloc[idx,2]

    # Se extrae la velocidad del viento y la pendiente utilizando las coordenadas y el valor más cercano
    pendiente = tif_pendiente.sel(
        y=lat,x=lon,
        method='nearest'
    ).values[0]

    viento = tif_viento.sel(
        y=lat,x=lon,
        method='nearest'
    ).values[0]
    # Actualizacion especies
    # Se extrae el score de las especies de aves intersectando los municipios y la capa shapefile
    punto = Point(lon, lat)	
    score_especies = gdf_especies[gdf_especies.intersects(punto)].iloc[0,0]
    # Actualización de los datos
    fuente_a.at[idx, 'vel_viento'] = viento
    fuente_a.at[idx, 'pendiente'] = pendiente
    fuente_a.at[idx, 'aves'] = score_especies

In [51]:
# Extraccion IDI e IPM
fuente_b = base.copy()

for idx,sitio in enumerate(base['sitio']):

    # busqueda del municipio
    mask = mnpios_col['MpNombre'] == sitio
    gdf_filtro = mnpios_col[mask]

    # extraccion informacion IDI e IPM
    idi = gdf_filtro.iloc[0,-1]
    ipm = gdf_filtro.iloc[0,-2]

    fuente_b.at[idx, 'idi'] = idi
    fuente_b.at[idx, 'ipm'] = ipm

In [52]:
# Creacion de geodataframe con las fuentes de datos
lats = base['lat']
lons = base['lon']
geo = gpd.points_from_xy(x=lons,y=lats)
fuente_a_4326 = gpd.GeoDataFrame(
    data=fuente_a,
    geometry=geo,
    crs=crs_reference
)
fuente_b_4326 = gpd.GeoDataFrame(
    data=fuente_b,
    geometry=geo,
    crs=crs_reference
)

## 3. Generar la malla (H3)

Se crea una malla hexagonal que cubre la zona de estudio. Cada celda tiene un código único y
tamaño casi igual, lo que hace comparables los sitios.

In [53]:
RES = 5
zona = {"type": "Polygon", "coordinates": [[
    [-75.2, 8.0], [-71.5, 8.0], [-71.5, 12.2], [-75.2, 12.2], [-75.2, 8.0]]]}
malla = list(h3.geo_to_cells(zona, RES))   # el poligono va en orden GeoJSON [lon, lat]
print("celdas en la malla:", len(malla))

celdas en la malla: 788


## 4. Mapear las fuentes sobre la malla

A cada sitio se le asigna su celda H3 y, por celda, se resumen los criterios con el promedio.
Así las dos fuentes quedan unidas en una sola tabla por celda.

In [ ]:
def celda(lat, lon):
    return h3.latlng_to_cell(lat, lon, RES)

# lat/lon de la fuente A ya reproyectada
fa = fuente_a_4326.copy()
fa["lat"] = fa.geometry.y; fa["lon"] = fa.geometry.x
fa["h3"] = [celda(la, lo) for la, lo in zip(fa["lat"], fa["lon"])]
fb = fuente_b_4326.copy()
fb["h3"] = [celda(la, lo) for la, lo in zip(fb["lat"], fb["lon"])]

agg_a = fa.groupby("h3", as_index=False).agg(sitio=("sitio", lambda x: ", ".join(x)),
                                             vel_viento=("vel_viento", "mean"),
                                             pendiente=("pendiente", "mean"),
                                             aves=('aves',"mean"))

agg_b = fb.groupby("h3", as_index=False).agg(idi=("idi", "mean"),
                                             ipm=('ipm','mean'))

celdas = agg_a.merge(agg_b, on="h3", how="outer")
celdas

,h3,sitio,vel_viento,pendiente,aves,idi,ipm
0,856601abfffffff,Aguachica,3.781898,0.831025,5.00,55.75,37.0
1,8566220bfffffff,Barranquilla,5.756345,0.862486,5.00,84.07,17.4
2,856622b3fffffff,Santa Marta,7.209671,0.515623,1.25,85.04,24.4
3,856623b7fffffff,Valledupar,6.581703,0.762131,5.00,79.35,30.5
4,85662527fffffff,Maicao,6.795211,0.234419,5.00,52.64,60.0
5,8566257bfffffff,Riohacha,6.575811,0.000000,1.25,58.81,45.1
6,856625abfffffff,Manaure,7.215685,0.000000,5.00,76.02,86.7
7,8567566bfffffff,Uribia,8.458921,0.087739,5.00,66.23,92.2


## 5. Visualizar

Se dibujan las celdas con dato, coloreadas por viento, para verificar que la unificación y el
mapeo quedaron bien.

In [ ]:
import branca.colormap as cm
cmap = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["vel_viento"].min(), vmax=celdas["vel_viento"].max())
cmap.caption = "Viento (m/s)"
m = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=cmap(r["vel_viento"]), fill_opacity=0.75,
                   tooltip=f"{r['sitio']}: viento {r['vel_viento']:.1f} m/s").add_to(m)
cmap.add_to(m)
m

## 6. Decidir: AHP y mapa de idoneidad

Con todo en la malla, se aplica AHP (Módulo 3) para combinar los criterios en un puntaje.
Criterios: viento (beneficio), distancia a vía (costo), radiación (beneficio).

In [32]:
crit = {
    "vel_viento": "beneficio", 
    "pendiente": "costo", 
    "aves": "costo",
    "idi": "beneficio",
    "ipm": "costo",
}


def nz(col, sentido):
    v = celdas[col].astype(float); z = (v - v.min()) / (v.max() - v.min())
    return z if sentido == "beneficio" else 1 - z
norm = pd.DataFrame({c: nz(c, s) for c, s in crit.items()})

# pesos AHP (matriz de comparacion por pares, Modulo 3)
A = np.array([
    [  1,   3,   5,   2,   4],  # vel_viento
    [1/3,   1,   2, 1/2,   2],  # pendiente
    [1/5, 1/2,   1, 1/4, 1/2],  # aves
    [1/2,   2,   4,   1,   3],  # idi
    [1/4, 1/2,   2, 1/3,   1]   # ipm
], dtype=float)

w = (A / A.sum(0)).mean(1)
pesos = dict(zip(crit.keys(), w))
print("pesos AHP:", {c: round(v, 3) for c, v in pesos.items()})

celdas["idoneidad"] = sum(norm[c] * pesos[c] for c in crit)
celdas.sort_values("idoneidad", ascending=False)[["h3", "idoneidad"]].round(3).head()

pesos AHP: {'vel_viento': np.float64(0.419), 'pendiente': np.float64(0.149), 'aves': np.float64(0.068), 'idi': np.float64(0.264), 'ipm': np.float64(0.1)}


,h3,idoneidad
2,856622b3fffffff,0.790
7,8567566bfffffff,0.663
6,856625abfffffff,0.654
5,8566257bfffffff,0.580
3,856623b7fffffff,0.568


In [33]:
mapa = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["idoneidad"].min(), vmax=celdas["idoneidad"].max())
mapa.caption = "Idoneidad (AHP)"
m2 = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=mapa(r["idoneidad"]), fill_opacity=0.8,
                   tooltip=f"{r['sitio']}: idoneidad {r['idoneidad']:.2f}").add_to(m2)
mapa.add_to(m2)
m2

### Guardar el resultado

El resultado se guarda en el formato de trabajo del proyecto: `.h3.parquet` (celda H3 e idoneidad).

In [ ]:
resultado = celdas[["h3", "sitio", "vel_viento", "pendiente", "aves","idi", "ipm","idoneidad"]].rename(
    columns={"h3": "h3_index"})
resultado.to_parquet("resultado_idoneidad.h3.parquet", index=False)
print("guardado resultado_idoneidad.h3.parquet")
resultado.sort_values("idoneidad", ascending=False).round(3).head()

guardado resultado_idoneidad.h3.parquet


,h3_index,sitio,vel_viento,pendiente,aves,idi,ipm,idoneidad
2,856622b3fffffff,Santa Marta,7.210,0.516,1.25,85.04,24.4,0.790
7,8567566bfffffff,Uribia,8.459,0.088,5.00,66.23,92.2,0.663
6,856625abfffffff,Manaure,7.216,0.000,5.00,76.02,86.7,0.654
5,8566257bfffffff,Riohacha,6.576,0.000,1.25,58.81,45.1,0.580
3,856623b7fffffff,Valledupar,6.582,0.762,5.00,79.35,30.5,0.568


## 7. Evaluación de Idoneidad AHP a Escala Regional Continua sobre Malla H3

En las secciones anteriores, el análisis AHP se limitó a la evaluación puntual de los centroides de los municipios seleccionados. Para escalar el análisis a una **perspectiva territorial continua**, en esta sección extenderemos la metodología a la totalidad de la malla hexagonal H3 (~788 celdas) que recubre el norte de Colombia.

Este enfoque permite:
- Superar los límites administrativos rígidos mediante una teselación espacial homogénea.
- Capturar la variabilidad intra e intermunicipal de las variables físicas, ambientales y socioeconómicas.
- Generar un mapa continuo de idoneidad eólica (*Choropleth*) para la toma de decisiones territoriales.

### 7.1. Geometrización de la Malla H3 a GeoDataFrame
Las celdas de la malla regional están almacenadas como identificadores de cadena (índices H3). Para realizar operaciones espaciales vectoriales y ráster, primero convertimos cada índice H3 en un polígono (`shapely.geometry.Polygon`) y construimos un `GeoDataFrame` con el sistema de referencia de coordenadas geográficas WGS84 (`EPSG:4326`).

In [34]:
def h3_to_polygon(cell_id):
  """Convierte el ID de un celda H3 a un objeto Polygon de Shapely (EPSG:4326)."""
  # h3.cell_to_boundary retorna vértices en orden (lat, lon)
  boundary = h3.cell_to_boundary(cell_id)
  # Shapely requiere orden (lon, lat) -> (x, y)
  return Polygon([(lon, lat) for lat, lon in boundary])


# Creación del GeoDataFrame regional
gdf_malla = gpd.GeoDataFrame(
    {'h3': malla},
    geometry=[h3_to_polygon(cell) for cell in malla],
    crs=crs_reference,  # EPSG:4326
)

print(f'Malla regional convertida a GeoDataFrame: {len(gdf_malla)} celdas H3.')

Malla regional convertida a GeoDataFrame: 788 celdas H3.


### 7.2. Extracción y Agregación Espacial de Criterios
Dado que los insumos provienen de distintas fuentes y formatos (ráster y vectorial), se aplican dos estrategias de agregación espacial para resumir la información dentro de cada hexágono:

1. **Estadísticas Zonales (`zonal_stats`):** Extrae el valor promedio de la velocidad del viento ($m/s$) y la pendiente (%) desde los archivos Ráster (`.tif`).
2. **Uniones Espaciales (`sjoin`):** Calcula el promedio de la riqueza de especies de aves y de los índices socioeconómicos (IDI e IPM) a partir de las capas vectoriales.

Finalmente, se filtran las celdas sin cobertura de datos (e.g., zonas marítimas o fuera del dominio del estudio).

In [54]:
# ------------------------------------------------------------------------------
# 7.2. Extracción de datos por Hexágono mediante la Media Espacial
# ------------------------------------------------------------------------------

# A. Velocidad del Viento (Ráster) -> Media por hexágono
stats_viento = zonal_stats(
    gdf_malla,
    tif_viento.values[0],
    affine=tif_viento.rio.transform(),
    stats=['mean'],
    nodata=np.nan,
)
gdf_malla['vel_viento'] = [s['mean'] for s in stats_viento]

# B. Pendiente (Ráster) -> Media por hexágono
stats_pendiente = zonal_stats(
    gdf_malla,
    tif_pendiente.values[0],
    affine=tif_pendiente.rio.transform(),
    stats=['mean'],
    nodata=np.nan,
)
gdf_malla['pendiente'] = [s['mean'] for s in stats_pendiente]

# C. Riqueza de Aves (Vectorial) -> Media por hexágono
aves_join = gpd.sjoin(
    gdf_malla[['h3', 'geometry']],
    gdf_especies,
    how='left',
    predicate='intersects',
)
col_aves = gdf_especies.columns[0]  # Columna con el score de especies de aves
mean_aves = aves_join.groupby('h3')[col_aves].mean()
gdf_malla['aves'] = gdf_malla['h3'].map(mean_aves)

# D. Variables Socioeconómicas IDI e IPM (Vectorial Municipios) -> Media por hexágono
socio_join = gpd.sjoin(
    gdf_malla[['h3', 'geometry']],
    mnpios_col[['idi', 'ipm', 'geometry']],
    how='left',
    predicate='intersects',
)
socio_mean = socio_join.groupby('h3')[['idi', 'ipm']].mean()
gdf_malla['idi'] = gdf_malla['h3'].map(socio_mean['idi'])
gdf_malla['ipm'] = gdf_malla['h3'].map(socio_mean['ipm'])

# Filtrar celdas marítimas o fuera de la cobertura de datos
gdf_ahp = gdf_malla.dropna(
    subset=['vel_viento', 'pendiente', 'aves', 'idi', 'ipm']
).copy()
print(
    f'Celdas continentales con información completa: {len(gdf_ahp)} de'
    f' {len(gdf_malla)}'
)

Celdas continentales con información completa: 465 de 788


### 7.3 y 7.4. Normalización Criterial y Cálculo Vectorial del Índice AHP
Para combinar variables con diferentes escalas y unidades de medida (m/s, %, índices, etc.), se aplica una **normalización Min-Max** en el rango $[0, 1]$:

* **Criterios de Beneficio (A maximizar):** Viento, IDI e IPM. Un valor más alto incrementa la idoneidad.
* **Criterios de Costo / Restricción (A minimizar):** Pendiente y Riqueza de Aves. Un valor más bajo incrementa la idoneidad (menor costo constructivo y menor impacto ambiental).

$$S_i = \sum_{j=1}^{n} w_j \cdot x_{ij}$$

Donde $w_j$ representa el peso relativo AHP asignado al criterio $j$ y $x_{ij}$ el valor normalizado del hexágono $i$.

In [56]:
# ------------------------------------------------------------------------------
# 7.3. Normalización Min-Max de Criterios [Escala 0 a 1]
# ------------------------------------------------------------------------------


def min_max_norm(series, maximize=True):
  """Normaliza una serie entre 0 y 1.

  maximize=True : Mayor valor -> Mayor idoneidad (1.0) maximize=False: Menor
  valor -> Mayor idoneidad (1.0)
  """
  min_val, max_val = series.min(), series.max()
  if max_val == min_val:
    return series * 0
  if maximize:
    return (series - min_val) / (max_val - min_val)
  else:
    return (max_val - series) / (max_val - min_val)


# Criterios a Maximizar (a mayor valor, mejor para el proyecto)
gdf_ahp['norm_viento'] = min_max_norm(gdf_ahp['vel_viento'], maximize=True)
gdf_ahp['norm_idi'] = min_max_norm(gdf_ahp['idi'], maximize=True)
gdf_ahp['norm_ipm'] = min_max_norm(
    gdf_ahp['ipm'], maximize=True
)  # Priorizar áreas con mayor necesidad social

# Criterios a Minimizar (a menor valor, menor costo/impacto)
gdf_ahp['norm_pendiente'] = min_max_norm(
    gdf_ahp['pendiente'], maximize=False
)  # Terrenos planos
gdf_ahp['norm_aves'] = min_max_norm(
    gdf_ahp['aves'], maximize=False
)  # Menor impacto ambiental

# ------------------------------------------------------------------------------
# 7.4. Aplicación de Pesos AHP y Cálculo de Idoneidad Regional
# ------------------------------------------------------------------------------
# Pesos AHP definidos (suman 1.0)
pesos = {
    'norm_viento': 0.35,  # 35% Importancia del recurso eólico
    'norm_pendiente': 0.20,  # 20% Facilidad constructiva / topografía
    'norm_aves': 0.20,  # 20% Sensibilidad ambiental
    'norm_idi': 0.15,  # 15% Desempeño institucional local
    'norm_ipm': 0.10,  # 10% Impacto/Desarrollo social
}

gdf_ahp['idoneidad'] = (
    gdf_ahp['norm_viento'] * pesos['norm_viento']
    + gdf_ahp['norm_pendiente'] * pesos['norm_pendiente']
    + gdf_ahp['norm_aves'] * pesos['norm_aves']
    + gdf_ahp['norm_idi'] * pesos['norm_idi']
    + gdf_ahp['norm_ipm'] * pesos['norm_ipm']
)

gdf_ahp[['h3', 'vel_viento', 'pendiente', 'aves', 'idi', 'ipm', 'idoneidad']].head()

,h3,vel_viento,pendiente,aves,idi,ipm,idoneidad
1,856605c7fffffff,3.797645,0.324954,5.000000,44.670000,53.633333,0.344439
2,8566204bfffffff,10.407064,0.499450,4.719081,85.040000,24.400000,0.592354
3,85662317fffffff,2.862229,7.845862,4.898649,58.210000,59.400000,0.236583
4,856600abfffffff,3.071802,0.011972,5.000000,188.533107,57.275000,0.392802
5,85662567fffffff,5.149911,0.234557,5.000000,58.810000,45.100000,0.392428


### 7.5. Visualización Cartográfica Interactiva de Idoneidad
Para finalizar, se representa el índice de idoneidad compuesto mediante un mapa coroplético interactivo (`folium.Choropleth`). La paleta de color *Yellow-Orange-Red* (YlOrRd) resalta en tonos rojos intensos las zonas con mayor potencial eólico y menores restricciones, e incluye etiquetas emergentes (*Tooltips*) para inspeccionar las métricas individuales de cada celda.

In [57]:
# ------------------------------------------------------------------------------
# 7.5. Visualización del Mapa Regional Continuo de Idoneidad Eólica
# ------------------------------------------------------------------------------
mapa_regional = folium.Map(
    location=[10.5, -73.0], zoom_start=7, tiles="cartodbpositron"
)

# Capa de Choropleth para colorear cada hexágono según su idoneidad
folium.Choropleth(
    geo_data=gdf_ahp,
    name="Mapa de Idoneidad AHP Regional",
    data=gdf_ahp,
    columns=["h3", "idoneidad"],
    key_on="feature.properties.h3",
    fill_color="YlOrRd",  # Escala de Amarillo (Baja) a Rojo (Alta)
    fill_opacity=0.75,
    line_opacity=0.2,
    legend_name="Índice de Idoneidad Eólica AHP (0 - 1)",
).add_to(mapa_regional)

# Agregar cuadro desplegable / Tooltip al pasar el cursor sobre cada hexágono
tooltip = folium.GeoJsonTooltip(
    fields=["h3", "idoneidad", "vel_viento", "pendiente", "aves", "idi", "ipm"],
    aliases=[
        "H3 ID:",
        "Idoneidad AHP:",
        "Viento (m/s):",
        "Pendiente (%):",
        "Score Aves:",
        "IDI:",
        "IPM:",
    ],
    localize=True,
    sticky=False,
    labels=True,
    style="""
        background-color: #F0EFEF;
        border: 2px solid grey;
        border-radius: 3px;
        box-shadow: 3px 3px 3px rgba(0,0,0,0.2);
    """,
)

folium.GeoJson(
    gdf_ahp,
    style_function=lambda x: {"fillColor": "#00000000", "color": "#00000000"},
    tooltip=tooltip,
).add_to(mapa_regional)

mapa_regional

## Actividad individual

Modifique los pesos de la matriz AHP, por ejemplo dándole más importancia a la cercanía a vías,
y observe cómo cambia el mapa de idoneidad.

Resultado esperado: con los pesos actuales domina el viento, y Uribia y Manaure quedan arriba
pese a estar lejos de vías. Al subir el peso de la cercanía a vías, los sitios bien conectados
como Riohacha o Barranquilla escalan posiciones y los de la alta Guajira bajan. Con los mismos
datos y otra prioridad, el resultado cambia: por eso justificar los pesos es parte de la decisión.

## Cierre

Recorrido completo del taller:

1. Fuentes en distintos CRS y formatos.
2. Unificación a un mismo sistema de coordenadas.
3. Malla H3 sobre la zona.
4. Mapeo de las fuentes a la malla.
5. Visualización.
6. Decisión con AHP y mapa de idoneidad.

Es el mismo flujo del proyecto real, a pequeña escala. Para llevarlo más lejos: más criterios,
mayor resolución en la malla, exclusiones (zonas donde no se puede) y datos reales por municipio.